# Лекция 16: LSTM + GARCH для KuCoin (NEAR)

Ноутбук в стиле референса «Тема_13_LSTM_и_нейронные_сети»: окна, признаки, LSTM-прогноз цены и GARCH-волатильность.
В конце экспортируем `latest_forecast_signal_kucoin_rl.json` для RL-бота.


## 1. Установка библиотек


In [ ]:
!pip -q install pandas numpy requests matplotlib tensorflow arch scikit-learn


## 2. Импорты и параметры


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import tensorflow as tf
from arch import arch_model
from sklearn.metrics import mean_absolute_error, mean_squared_error
SPOT_SYMBOL = 'NEAR-USDT'
FUTURES_SYMBOL = 'NEARUSDTM'
CANDLE_TYPE = '1min'
HISTORY_LIMIT = 1200
WINDOW = 30
FORECAST_HORIZON = 1
TRAIN_FRAC = 0.70
VALID_FRAC = 0.15
RANDOM_SEED = 42
tf.keras.utils.set_random_seed(RANDOM_SEED)


## 3. Загрузка OHLCV с KuCoin


In [ ]:
url = 'https://api.kucoin.com/api/v1/market/candles'
params = {'symbol': SPOT_SYMBOL, 'type': CANDLE_TYPE}
raw = requests.get(url, params=params, timeout=20).json()['data']

df = pd.DataFrame(raw, columns=['ts', 'open', 'close', 'high', 'low', 'volume', 'turnover'])
df = df.iloc[::-1].reset_index(drop=True)
for col in ['open', 'high', 'low', 'close', 'volume', 'turnover']:
    df[col] = df[col].astype(float)
df['ts'] = pd.to_datetime(df['ts'].astype(int), unit='s', utc=True)
df = df.tail(HISTORY_LIMIT).copy().set_index('ts')

print('history shape:', df.shape)
df.tail(3)



## 4. Фичи для LSTM (как в референсе)


In [ ]:
feat = df.copy()
feat['ret_1'] = feat['close'].pct_change(1)
feat['ret_3'] = feat['close'].pct_change(3)
feat['ma_5'] = feat['close'].rolling(5).mean()
feat['ma_10'] = feat['close'].rolling(10).mean()
feat['ma_20'] = feat['close'].rolling(20).mean()
feat['hl_range'] = (feat['high'] - feat['low']) / feat['close']
feat['oc_change'] = (feat['close'] - feat['open']) / feat['open']
feat['vol_chg_1'] = feat['volume'].pct_change(1)

feature_cols = [
    'close', 'volume', 'ret_1', 'ret_3',
    'ma_5', 'ma_10', 'ma_20', 'hl_range', 'oc_change', 'vol_chg_1'
]

data_feat = feat[feature_cols].dropna().copy()
print('feature matrix:', data_feat.shape)



## 5. Окна для обучения


In [ ]:
X_values = data_feat[feature_cols].to_numpy(dtype='float32')
y_values = data_feat['close'].to_numpy(dtype='float32')
dates = data_feat.index

X_all, y_all, idx_all = [], [], []
for i in range(WINDOW, len(X_values) - FORECAST_HORIZON + 1):
    X_all.append(X_values[i - WINDOW:i, :])
    target_pos = i + FORECAST_HORIZON - 1
    y_all.append(y_values[target_pos])
    idx_all.append(dates[target_pos])

X_all = np.asarray(X_all, dtype='float32')
y_all = np.asarray(y_all, dtype='float32')
idx_all = pd.DatetimeIndex(idx_all)

n = len(X_all)
train_end = int(n * TRAIN_FRAC)
valid_end = int(n * (TRAIN_FRAC + VALID_FRAC))

X_train, y_train = X_all[:train_end], y_all[:train_end]
X_valid, y_valid = X_all[train_end:valid_end], y_all[train_end:valid_end]
X_test, y_test = X_all[valid_end:], y_all[valid_end:]
idx_valid = idx_all[train_end:valid_end]
idx_test = idx_all[valid_end:]

x_mean = X_train.mean(axis=(0, 1), keepdims=True)
x_std = X_train.std(axis=(0, 1), keepdims=True) + 1e-8
y_mean = y_train.mean()
y_std = y_train.std() + 1e-8

X_train_s = ((X_train - x_mean) / x_std).astype('float32')
X_valid_s = ((X_valid - x_mean) / x_std).astype('float32')
X_test_s = ((X_test - x_mean) / x_std).astype('float32')

y_train_s = ((y_train - y_mean) / y_std).astype('float32')
y_valid_s = ((y_valid - y_mean) / y_std).astype('float32')
y_test_s = ((y_test - y_mean) / y_std).astype('float32')

print('train/valid/test:', X_train_s.shape, X_valid_s.shape, X_test_s.shape)



## 6. Обучение LSTM


In [ ]:
n_features = X_train_s.shape[2]

lstm_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(WINDOW, n_features)),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(1),
])

lstm_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse', metrics=['mae'])

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

history = lstm_model.fit(
    X_train_s,
    y_train_s,
    validation_data=(X_valid_s, y_valid_s),
    epochs=30,
    batch_size=32,
    verbose=0,
    callbacks=[early_stop],
    shuffle=False,
)

pd.DataFrame(history.history).plot(title='LSTM training history')
plt.grid(alpha=0.2)
plt.show()



## 7. Метрики и прогноз


In [ ]:
pred_valid_s = lstm_model.predict(X_valid_s, verbose=0).ravel()
pred_test_s = lstm_model.predict(X_test_s, verbose=0).ravel()

pred_valid = pred_valid_s * y_std + y_mean
pred_test = pred_test_s * y_std + y_mean

valid_mae = mean_absolute_error(y_valid, pred_valid)
valid_rmse = np.sqrt(mean_squared_error(y_valid, pred_valid))
valid_mape = np.mean(np.abs(y_valid - pred_valid) / np.maximum(np.abs(y_valid), 1e-12))

test_mae = mean_absolute_error(y_test, pred_test)
test_rmse = np.sqrt(mean_squared_error(y_test, pred_test))
test_mape = np.mean(np.abs(y_test - pred_test) / np.maximum(np.abs(y_test), 1e-12))

print('valid mae/rmse/mape:', valid_mae, valid_rmse, valid_mape)
print('test  mae/rmse/mape:', test_mae, test_rmse, test_mape)

last_window = data_feat[feature_cols].iloc[-WINDOW:].to_numpy(dtype='float32')
last_window_s = ((last_window[None, :, :] - x_mean) / x_std).astype('float32')
forecast_price_s = float(lstm_model.predict(last_window_s, verbose=0).ravel()[0])
forecast_price = float(forecast_price_s * y_std + y_mean)
spot_price = float(data_feat['close'].iloc[-1])
ret_hat = float(np.log(forecast_price / spot_price) / FORECAST_HORIZON)

print('spot_price:', spot_price)
print('forecast_price:', forecast_price)
print('ret_hat:', ret_hat)



## 8. GARCH-волатильность


In [ ]:
# sigma_hat ??????? ?????? ?? ??????????? returns, ? ?? ?? ???
returns = np.log(data_feat['close'] / data_feat['close'].shift(1)).dropna()
garch = arch_model(returns * 100, mean='Zero', vol='GARCH', p=1, q=1).fit(disp='off')
var_1 = float(garch.forecast(horizon=1).variance.iloc[-1, 0])
sigma_hat = float(max((var_1 ** 0.5) / 100, 1e-4))
# Kelly: z = ret_hat / sigma_hat^2
kelly_z = float(ret_hat / (sigma_hat**2 + 1e-8))
kelly_z = float(np.clip(kelly_z, -2.0, 2.0))
kelly_n_usdt = 1.5
kelly_fraction = 0.5
target_notional_usdt = float(kelly_n_usdt * kelly_fraction * kelly_z)
signal_direction = 'LONG' if forecast_price > spot_price else 'SHORT'
print('sigma_hat:', sigma_hat)
print('kelly_z:', kelly_z)
print('kelly_fraction:', kelly_fraction)
print('target_notional_usdt:', target_notional_usdt)
print('signal_direction:', signal_direction)


## 9. Экспорт JSON для RL-бота


In [ ]:
state = {
    'timestamp': str(data_feat.index[-1]),
    'asset': 'NEAR',
    'spot_symbol': SPOT_SYMBOL,
    'futures_symbol': FUTURES_SYMBOL,
    'spot_price': float(spot_price),
    'futures_price': float(spot_price),
    'ret_hat': float(ret_hat),
    'sigma_hat': float(sigma_hat),
    'z_score': float(kelly_z),
    'signal_direction': signal_direction,
    'signal_model': 'lstm_garch_ref',
    'window': int(WINDOW),
    'forecast_horizon': int(FORECAST_HORIZON),
    'forecast_price': float(forecast_price),
    'target_notional_usdt': float(target_notional_usdt),
    'valid_mae': float(valid_mae),
    'valid_rmse': float(valid_rmse),
    'valid_mape': float(valid_mape),
    'test_mae': float(test_mae),
    'test_rmse': float(test_rmse),
    'test_mape': float(test_mape),
    'candle_type': CANDLE_TYPE,
}
out_dir = Path('reports/kucoin_rl')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'latest_forecast_signal_kucoin_rl.json'
out_path.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved:', out_path)
print(out_path.read_text(encoding='utf-8'))


## 10. Команды для терминала


In [ ]:
print('SHADOW:')
print('python run_trade_signal.py --mode shadow --config config/micro_near_v1_1m.json --state-json reports/kucoin_rl/latest_forecast_signal_kucoin_rl.json')
print('\nLIVE:')
print('python run_trade_signal.py --run-real-order --config config/micro_near_v1_1m.json --state-json reports/kucoin_rl/latest_forecast_signal_kucoin_rl.json')
print('\nBUY NEAR spot:')
print('python run_trade_signal.py --run-real-order --config config/micro_near_v1_1m.json --state-json reports/kucoin_rl/latest_forecast_signal_kucoin_rl.json --force-action BUY_SPOT --spot-qty 0.1')
